# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by their @id
record_sets = list(dataset.record_sets)
print("Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            print(f"    - @id: {field['@id']} | name: {field.get('name', 'N/A')} | dataType: {field.get('dataType', 'N/A')}")
    print()
# If no record sets are listed, try records() with inferred @id
# Print sample records from each record set using @id
for rs in record_sets:
    print(f"Sample records for record set {rs['@id']}:")
    for i, record in enumerate(dataset.records(record_set=rs['@id'])):
        if i < 3:
            print(record)
        else:
            break
    print()

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

if dataframes:
    # Choose the first record set as default for demonstration
    main_record_set = record_set_ids[0]
    print(f"DataFrame columns for record set @id {main_record_set}:")
    print(dataframes[main_record_set].columns.tolist())
    print(dataframes[main_record_set].head())
else:
    print("No records available to extract.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers and grouping data.

In [ ]:
import numpy as np

# Select a numeric field for analysis (e.g., GAD-7 score, PHQ-9 score, PSQ score)
# Assume a field such as 'gad_7_score' or similar exists based on survey description
numeric_field = None
possible_numeric_fields = [
    col for col in dataframes[main_record_set].columns 
    if 'gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower() or 'score' in col.lower()
]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    # Default to any numerical column
    for col in dataframes[main_record_set].columns:
        if np.issubdtype(dataframes[main_record_set][col].dtype, np.number):
            numeric_field = col
            break

threshold = 10
if numeric_field:
    filtered_df = dataframes[main_record_set][dataframes[main_record_set][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Choose a field for grouping, e.g., 'gender', 'village', or 'level_of_education' (personalSensitiveInformation)
    group_field = None
    possible_group_fields = [
        'gender', 'village', 'level_of_education', 'age', 'marital_status'
    ]
    for col in possible_group_fields:
        if col in filtered_df.columns:
            group_field = col
            break
    if group_field:
        grouped_df = (
            filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        )
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set][numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouped_df exists, plot mean numeric_field by group_field
    if 'grouped_df' in locals():
        plt.figure(figsize=(6,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("Nothing to visualize as no numeric field was found.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset provides rich demographic and mental health assessment data for analysis.
- Using `mlcroissant`, we loaded, overviewed, and processed the survey responses referencing all entities by their `@id`.
- Data processing allowed us to filter, normalize, and group numeric measures such as GAD-7/PHQ-9 scores by sensitive demographic attributes.
- Visualizations showed potential distributions and patterns for key mental health indicators.

For further exploration, consider deeper analyses including missing data evaluation, more advanced statistical tests, and richer visualizations.